# Adaptive PyTorch Deep Learning Lab
RNN-first, observable architecture growth; TensorFlow/JAX are future plugins. Run from repo root.

In [ ]:
import torch
from adaptive.models import AdaptiveGRU, TinyCNN, parameter_count
from adaptive.train import run, dataset

## Tensor input and recurrent architecture
Input shape = `[batch, time, features]`. Our primary model maps four features per step to a hidden representation; each residual GRU block processes a sequence.

In [ ]:
model = AdaptiveGRU(width=16, max_blocks=3)
x, y = dataset(4)
print(x.shape, y.shape, model(x).shape)
print(model)
print("parameters:", parameter_count(model))

## Add a recurrent block without disrupting its initial output
The gate starts at zero, so its contribution is initially zero.

In [ ]:
with torch.no_grad(): before = model(x).clone()
model.grow()
with torch.no_grad(): after = model(x)
print("blocks:",len(model.blocks),"parameters:", parameter_count(model))
print("max difference at insertion:", (before-after).abs().max().item())

## Adaptive training
The controller grows when validation loss plateaus, or use a forced trigger only to demonstrate the mechanism.

In [ ]:
trained, history = run(epochs=12, force_grow_epoch=4, output="runs/notebook", width=16)
print("growth events:",[row for row in history if row["grew"]])

## CNN alternative
CNN consumes image-shaped tensors. It is a separate family using the same PyTorch ecosystem.

In [ ]:
cnn = TinyCNN()
print(cnn(torch.randn(2,1,16,16)).shape)
print(cnn)

## Optional future integrations
`adaptive/backends.py` contains commented backend registration points for TensorFlow and JAX. Neither is necessary for PyTorch tensor training. Deployment preview: `python deploy.py --deploy kubernetes --image YOUR_IMAGE`.